# 性能调优

## 概述

6.2解决了"写对"的问题，本节解决"写快"。看一个新场景：`./src/add_slow.py` 运行**完全正确**（输出`Sample add run success.`），但耗时可观——**正确≠高效**。它是第2章Add算子的一个"劣化变体"：单缓冲（`BUFFER_NUM=1`）+ 不切tile（`TILE_NUM=1`），整块数据"搬入→计算→搬出"完全串行。

凭肉眼很难判断慢在哪里：是搬运太慢？计算太慢？还是组织方式不对？本节用msprof op工具完成**采集→分析→优化→复测**的完整调优闭环：

1. 用 **msprof op**（上板模式）采集劣化版性能数据；
2. 用pandas读取CSV指标，判定瓶颈在哪个环节；
3. 改造为双缓冲流水实现；
4. 复测对比，验证优化收益。

---
# 1. 环境准备

初始化jupyter环境，并确认msprof工具可用（msprof随CANN工具链安装，环境变量生效后可直接调用）。本节的工作目录：`prof`（劣化版性能数据）、`prof2`（优化版性能数据）。


In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
os.makedirs("Sources/06.03", exist_ok=True)
print("环境初始化完成")

In [ ]:
# 确认msprof工具可用
!which msprof && msprof op --help | head -5

---
# 2. 性能调优方法论

### 2.1 理论性能

调优前先明确"多快才算快"。理论性能是算子在当前硬件上的理想目标，由硬件规格决定：

- **搬运带宽上限**：MTE2/MTE3搬运通路的带宽决定了数据搬入搬出的最短时间；
- **计算吞吐上限**：Vector/Cube单元的指令吞吐决定了纯计算的最短时间。

实测耗时与理论值的差距，就是优化空间。

### 2.2 瓶颈判定标准

拿到性能数据后，两类信号指向瓶颈：

1. **耗时最长的环节**：搬运、计算、同步阻塞中哪个占比最高；
2. **实际值与理论值差距大的环节**：差距越大，该环节越有优化价值。

### 2.3 调优工作流

<img src="./images/profiling_workflow.png" alt="性能调优闭环工作流" width="900px">

采集→分析→定位优化→复测，四步循环，直到实测逼近理论或收益递减为止。

---
# 3. msprof op上板采集

### 3.1 采集命令

msprof op用于在实际NPU环境（上板）采集单算子程序的性能数据：

```shell
msprof op --output=./prof python3 ./src/add_slow.py -r NPU
```

`--output`指定输出目录，`python3 xxx.py -r NPU`是被采集的算子验证程序。

### 3.2 输出目录结构

采集完成后，`prof`目录下生成带时间戳的`OPPROF_*`子目录：

```text
prof/
└── OPPROF_{timestamp}_{random}
    ├── dump
    ├── ArithmeticUtilization.csv    # 数值计算单元利用率
    ├── L2Cache.csv                  # L2 Cache命中情况
    ├── Memory.csv                   # 各级存储读写带宽与耗时
    ├── MemoryL0.csv                 # L0存储访问
    ├── MemoryUB.csv                 # UB存储访问
    ├── OpBasicInfo.csv              # 算子基本信息（任务类型/核数/aiv_time等）
    ├── PipeUtilization.csv          # 各指令流水线利用率与耗时分解
    ├── ResourceConflictRatio.csv    # 资源冲突比例
    └── visualize_data.bin           # 可视化数据（导入MindStudio Insight）
```

### 3.3 关键CSV含义

| CSV文件 | 关注点 |
| --- | --- |
| OpBasicInfo.csv | 算子类型、block数、**aiv_time**（AI Vector核心总耗时） |
| PipeUtilization.csv | **MTE2/Vector/MTE3各流水线耗时与利用率**（判定串行/重叠的核心依据） |
| Memory.csv / MemoryUB.csv | 搬运量与带宽利用率（对照理论带宽） |
| ArithmeticUtilization.csv | 计算单元利用率（对照理论算力） |
| ResourceConflictRatio.csv | 同步等待/资源冲突占比 |

`visualize_data.bin`可导入MindStudio Insight，以图形化方式查看通算流水图、内存热力图、Roofline瓶颈分析图等。

In [ ]:
# 上板采集劣化版add_slow性能数据
!rm -rf prof && msprof op --output=./prof python3 ./src/add_slow.py -r NPU

---
# 4. 分析性能数据

用pandas读取PipeUtilization.csv（目录名含时间戳，用glob通配）：

关注两处：

1. **aiv_time**（AI Vector核心总耗时）：劣化版的单核总耗时；
2. **各流水线耗时分解**：MTE2（搬入）、Vector（计算）、MTE3（搬出）三段的耗时。

**瓶颈判定**：劣化版`BUFFER_NUM=1、TILE_NUM=1`，三段指令**完全串行**——总耗时≈搬入+计算+搬出之和，任何时刻只有一条流水线在工作（呼应2.5节的"串行 vs 双缓冲"理论：N个数据块串行总耗时3N×T，双缓冲约(N+2)×T）。

这属于"组织方式"瓶颈而非"带宽/算力"瓶颈：搬运和计算各自都不慢，但**互相等待**导致时间白白流失。对应的优化方向明确——让三段流水重叠起来。

In [ ]:
# 读取劣化版PipeUtilization数据
import pandas as pd, glob
csv_file = glob.glob("prof/*/PipeUtilization.csv") or glob.glob("prof/*/PIPE*Utilization*.csv")
df_slow = pd.read_csv(csv_file[0])
df_slow

---
# 5. 针对性优化

将`add_slow.py`（手动串行单buffer）改造为`add_framework.py`（TPipe/TQue框架版双缓冲流水）：

### 5.1 优化点对比

| 维度 | add_slow.py（劣化） | add_framework.py（优化） |
| --- | --- | --- |
| BUFFER_NUM | 1（单缓冲） | 2（双缓冲） |
| TILE_NUM | 1（整块一次处理） | 8（切8个tile） |
| 同步方式 | 手动set_flag/wait_flag串行 | TQue的enque/deque自动流水 |
| 执行模式 | 搬入→计算→搬出依次执行 | 从第2个tile起三段重叠 |

### 5.2 关键代码对比

劣化版（手动串行——每拍都显式等待上一段完成）：

```python
for i in range(TILE_NUM * BUFFER_NUM):
    asc.data_copy(x_local[...], x_gm[i * tile_length:], tile_length)   # 搬入
    asc.set_flag(asc.HardEvent.MTE2_V, buf_id); asc.wait_flag(...)     # 等搬入完成
    asc.add(z_local[...], x_local[...], y_local[...], tile_length)     # 计算
    asc.set_flag(asc.HardEvent.V_MTE3, buf_id); asc.wait_flag(...)     # 等计算完成
    asc.data_copy(z_gm[i * tile_length:], z_local[...], tile_length)   # 搬出
```

优化版（TQue队列——框架自动管理缓冲区复用与流水调度）：

```python
for i in range(TILE_NUM * BUFFER_NUM):
    copy_in(i, x_gm, y_gm, in_queue_x, in_queue_y, tile_length)   # enque入队
    compute(z_gm, in_queue_x, in_queue_y, out_queue_z, tile_length) # deque/计算/enque
    copy_out(i, z_gm, out_queue_z, tile_length)                   # deque出队
```

`alloc_tensor/enque/deque/free_tensor`的队列语义隐藏了手动flag同步，硬件上两块buffer轮流使用，搬入(i+1)与计算(i)、搬出(i-1)时间上重叠。

先运行确认优化版功能正确：


In [ ]:
# 运行优化版确认功能正确
!python3 ./src/add_framework.py -r NPU

---
# 6. 复测对比

对优化版重新采集，拼接两版关键指标对比：

**预期结论**：数据量与核数完全一致（唯一变量=流水组织），优化版aiv_time显著低于劣化版——收益来自三段流水重叠，而非任何单环节的提速。这正是第4节瓶颈分析给出的方向。

In [ ]:
# 采集优化版性能数据并对比
!rm -rf prof2 && msprof op --output=./prof2 python3 ./src/add_framework.py -r NPU

In [ ]:
# 拼接优化前后PipeUtilization关键指标对比
import pandas as pd, glob

def read_pipe(d):
    f = glob.glob(d + "/*/PipeUtilization.csv") or glob.glob(d + "/*/PIPE*Utilization*.csv")
    return pd.read_csv(f[0])

df_fast = read_pipe("prof2")
# 取两版共有列做对比（具体列名以实际CSV为准，重点关注各流水线耗时/利用率列）
common = [c for c in df_slow.columns if c in df_fast.columns]
pd.concat([df_slow[common].add_prefix("slow_"), df_fast[common].add_prefix("fast_")], axis=1)

---
# 7. 课后练习

### 选择题

**1.** `msprof op --output=./prof python3 xxx.py -r NPU`中`--output`参数的作用是？

- A. 指定算子运行backend
- B. 指定性能数据输出目录
- C. 指定输出文件格式
- D. 指定采集的流水线类型

**2.** 判定性能瓶颈点的依据是？

- A. 耗时最长、且实际值与理论值差距大的环节
- B. 代码行数最多的函数
- C. printf打印最多的语句
- D. 编译告警数量

**3.** PipeUtilization.csv中MTE2/Vector/MTE3三条流水线耗时均较高且近似相等，最可能说明？

- A. 算子是访存密集型，应减少数据量
- B. 三段串行执行，应优化流水组织让其重叠
- C. 计算单元频率太低
- D. 数据类型选择不当

### 实践题

将`add_slow.py`的`USE_CORE_NUM`从8改为4（其余不变），重新用msprof op采集，观察并解释aiv_time与各流水线耗时的变化。

执行以下代码查看答案：


In [ ]:
!cat ./answer/06.03_answer.txt